**1.   Import các thư viện cần thiết**









In [ ]:
!pip install gensim

In [ ]:
import pandas as pd
import numpy as np
import json
import pickle
from gensim.models import KeyedVectors
from gensim.models.fasttext import load_facebook_model
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, SpatialDropout1D, Bidirectional, BatchNormalization, Layer
from tensorflow.keras import backend as K
from tensorflow.keras.callbacks import ReduceLROnPlateau

**2. Chuẩn hóa dataset và chuẩn bị dữ liệu để train**

**2.1. Chuẩn hóa dataset lại thành 2 cột (chỉ chạy lần đầu)**



In [ ]:
#Đọc dữ liệu từ file dataset cũ
df = pd.read_csv("/content/drive/MyDrive/SIC/dataset/Data_Completed.csv")
#Drop các cột có dữ liệu NaN
df = df.dropna(subset=['tokenized_text'])
# Tạo DataFrame mới
df_new = df[['tokenized_text', 'label']].copy()
df_new.rename(columns={'tokenized_text': 'segment_text'}, inplace=True)
# Lưu CSV mới
df_new.to_csv("/content/drive/MyDrive/SIC/dataset/data_for_lstm.csv", index=False, encoding='utf-8-sig')

**2.2. Encode label (chỉ chạy lần đầu)**

In [ ]:
#Encode label
le = LabelEncoder()
df['label_int'] = le.fit_transform(df['label'])
# label → int
label_to_int = dict(zip(le.classes_, le.transform(le.classes_)))

# int → label
int_to_label = {v: k for k, v in label_to_int.items()}

#Ép kiểu từ int32 hoặc int64 do LabelEncoder trả về thành int để lưu trong json
label_to_int = {k: int(v) for k, v in label_to_int.items()}
int_to_label = {int(k): v for k, v in int_to_label.items()}

# Gộp thành dict để lưu JSON
mapping_data = {
    "label_to_int": label_to_int,
    "int_to_label": int_to_label
}

# Lưu mapping vào JSON
with open("/content/drive/MyDrive/SIC/mapping/label_mapping.json", "w", encoding="utf-8") as f:
    json.dump(mapping_data, f, ensure_ascii=False, indent=4)

print("Đã lưu mapping vào label_mapping.json")


Đã lưu mapping vào label_mapping.json


**2.3. Đọc dataset mới và label mapping**

In [ ]:
#Đọc dữ liệu từ file đã chuẩn hóa
df = pd.read_csv("/content/drive/MyDrive/SIC/dataset/data_for_lstm.csv", encoding='utf-8-sig')
df = df[['segment_text', 'label']]
# Load mapping từ JSON
with open("/content/drive/MyDrive/SIC/mapping/label_mapping.json", "r", encoding="utf-8") as f:
    mapping_data = json.load(f)

int_to_label = mapping_data["int_to_label"]

**2.4. Tách tập train, test, validate random**

In [ ]:
#Chuyển label thành số từ label mapping
label_to_int = mapping_data["label_to_int"]

## Tạo cột label_int
df["label_int"] = df["label"].map(label_to_int)

#Tách data và label
X = df["segment_text"]
y = df["label_int"]

#Lần 1: tách test từ dataset
X_temp, X_test, y_temp, y_test = train_test_split(
    X,
    y,
    test_size=0.15,
    stratify=y,
    random_state=42
)

# Lần 2: tách validation từ phần còn lại
X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=0.1765,
    stratify=y_temp,
    random_state=42
)
print("Train:", y_train.value_counts(normalize=True))
print("Validation:", y_val.value_counts(normalize=True))
print("Test:", y_test.value_counts(normalize=True))


Train: label_int
0    0.333524
2    0.333238
1    0.333238
Name: proportion, dtype: float64
Validation: label_int
0    0.333333
1    0.333333
2    0.333333
Name: proportion, dtype: float64
Test: label_int
0    0.333482
1    0.333482
2    0.333037
Name: proportion, dtype: float64


**2.5. Data Tokenization**

In [ ]:
#Tokenize cho tập train dùng train model
max_features = 12000
tokenizer = Tokenizer(num_words = max_features, oov_token='OOV')
tokenizer.fit_on_texts(X_train)
#Lưu tokenizor
with open("/content/drive/MyDrive/SIC/tokenizor/tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)
print('Đã lưu tokenizor')
#Chuyển tất cả các tập về sequence
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq = tokenizer.texts_to_sequences(X_val)
X_test_seq = tokenizer.texts_to_sequences(X_test)

Đã lưu tokenizor


**2.6. Padding data về cùng sequence sau khi tokenize**

In [ ]:
#Tính độ dài tối đa để padding
##Lấy độ dài của các dòng trong dataset
lengths = [len(seq) for seq in X_train_seq]
##Số từ tối đa của 99% data trong dataset
max_len = (int) (np.percentile(lengths, 90))
print('Max lenght of sequence: ', max_len)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post')
X_val_pad = pad_sequences(X_val_seq, maxlen=max_len, padding='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post')

Max lenght of sequence:  21


**3. Build Model**

In [ ]:
class AttentionLayer(Layer):
    def __init__(self, **kwargs):
        super(AttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        # input_shape = (batch_size, time_steps, features)
        self.W = self.add_weight(name='att_weight',
                                 shape=(input_shape[-1],),
                                 initializer='random_normal',
                                 trainable=True)
        self.built = True
        super(AttentionLayer, self).build(input_shape)

    def call(self, inputs):
        # inputs: (batch, time, features)
        e = K.tanh(K.dot(inputs, K.expand_dims(self.W)))   # (batch, time, 1)
        a = K.softmax(e, axis=1)                           # (batch, time, 1)
        output = K.sum(inputs * a, axis=1)                 # (batch, features)
        return output


In [ ]:
max_features = min(len(tokenizer.word_index) + 1, max_features)
lstm_out = 128
embed_dim = 300
# 1) Load .vec file
ft_model = KeyedVectors.load_word2vec_format(
    '/content/drive/MyDrive/SIC/pretrained/cc.vi.300.sub.vec',
    binary=False
)

# 2) Chuẩn bị tham số
embedding_matrix = np.zeros((max_features, embed_dim))

# 3) Điền embedding matrix
for word, i in tokenizer.word_index.items():
    if i >= max_features:
        continue
    # kiểm tra tồn tại trong vocab
    if word in ft_model.key_to_index:
        embedding_matrix[i] = ft_model[word]

model = Sequential([
  Embedding(
    input_dim=max_features,
    output_dim=embed_dim,
    weights=[embedding_matrix],
    trainable=False
  ),
  SpatialDropout1D(0.4),
  Bidirectional(LSTM(lstm_out, return_sequences=True, dropout=0.5)),
  AttentionLayer(),
  LSTM(lstm_out, dropout=0.5),
  BatchNormalization(),
  Dense(128, activation='relu'),
  Dropout(0.5),
  BatchNormalization(),
  Dense(64, activation='relu'),
  Dropout(0.5),
  BatchNormalization(),
  Dense(32, activation='relu'),
  Dropout(0.5),
  BatchNormalization(),
  Dense(3, activation='softmax')
])
model.compile('adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.build(input_shape=(64, max_len))
model.summary()


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_6 (Embedding)         │ (128, 21, 300)         │     1,176,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d_6             │ (128, 21, 300)         │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_5 (Bidirectional) │ (128, 21, 256)         │       439,296 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ attention_layer_4               │ (128, 256)             │           256 │
│ (AttentionLayer)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_20          │ (128, 256)             │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_20 (Dense)                │ (128, 128)             │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_15 (Dropout)            │ (128, 128)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_21          │ (128, 128)             │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_21 (Dense)                │ (128, 64)              │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_16 (Dropout)            │ (128, 64)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_22          │ (128, 64)              │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_22 (Dense)                │ (128, 32)              │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_17 (Dropout)            │ (128, 32)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_23          │ (128, 32)              │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_23 (Dense)                │ (128, 3)               │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,661,403 (6.34 MB)

 Trainable params: 483,843 (1.85 MB)

 Non-trainable params: 1,177,560 (4.49 MB)

**4. Train Model**

In [ ]:
batch_size = 128
epochs = 100

# Khởi tạo callback
## EarlyStopping để dừng huấn luyện sớm nếu không cải thiện
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3, # Dừng lại sau 3 epochs không cải thiện
    restore_best_weights=True
)

## ReduceLROnPlateau giảm learning rate khi mô hình không cải thiện
reduce_lr = ReduceLROnPlateau(
  monitor='val_loss',
  factor=0.5,
  patience=2, # Giảm lại sau 2 epochs không cải thiện
  min_lr=1e-5
)

# Huấn luyện mô hình với dữ liệu và các callback đã khởi tạo
history = model.fit(
    X_train_pad,
    y_train,
    epochs=epochs,
    batch_size=batch_size,
    validation_data=(X_val_pad, y_val),
    verbose = 1,
    callbacks=[early_stop, reduce_lr]
)


Epoch 1/100
82/82 ━━━━━━━━━━━━━━━━━━━━ 27s 219ms/step - accuracy: 0.3425 - loss: 1.4926 - val_accuracy: 0.3333 - val_loss: 1.1007 - learning_rate: 0.0010
Epoch 2/100
82/82 ━━━━━━━━━━━━━━━━━━━━ 23s 250ms/step - accuracy: 0.3695 - loss: 1.2803 - val_accuracy: 0.3961 - val_loss: 1.0848 - learning_rate: 0.0010
Epoch 3/100


KeyboardInterrupt: 

**5. Test Model**